# Dự án Thực tập VNPT AI - Data Analyst
**Mục tiêu:** Tối ưu hoá dữ liệu hệ thống Chat (ETL, Time-Bucketing, NLP)

Notebook này mô phỏng toàn bộ chu trình xử lý dữ liệu từ Phase 0 đến Phase 4 (Tuần 8) trực tiếp trên Google Colab. 
Thay vì dùng Docker như máy cục bộ, toàn bộ hệ thống Database & Big Data Engine (Cassandra + PySpark) sẽ được dựng thẳng trong môi trường Colab.

## 1. Cài đặt Môi trường (Cassandra & Spark)

In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget https://archive.apache.org/dist/cassandra/4.1.7/apache-cassandra-4.1.7-bin.tar.gz
!tar -xzf apache-cassandra-4.1.7-bin.tar.gz
!pip install -q pyspark==3.5.1 underthesea cassandra-driver matplotlib pandas

import os
import time
import subprocess

# Set JVM limits to prevent Colab from crashing
os.environ["MAX_HEAP_SIZE"] = "512M"
os.environ["HEAP_NEWSIZE"] = "100M"
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

print("Đang khởi động Cassandra Database (khoảng 30 giây)...")
subprocess.Popen(["./apache-cassandra-4.1.7/bin/cassandra", "-R"])
time.sleep(30)

## 2. Sinh dữ liệu giả lập (Phase 0) & Khởi tạo Schema (Phase 2)
Dùng chính Cassandra làm Database đích (thay cho ScyllaDB) để demo tính năng Time-bucketing.

In [ ]:
from cassandra.cluster import Cluster
from cassandra.policies import DCAwareRoundRobinPolicy
from cassandra.concurrent import execute_concurrent_with_args
import random
import time
from datetime import datetime, timedelta

print("Đợi Cassandra khởi động hoàn toàn và nhận kết nối...")
for i in range(15):
    try:
        cluster = Cluster(['127.0.0.1'], port=9042, load_balancing_policy=DCAwareRoundRobinPolicy(local_dc='datacenter1'))
        session = cluster.connect()
        print("Kết nối Cassandra thành công!")
        break
    except Exception as e:
        print(f"Thử lại lần {i+1}/15 sau 5 giây...")
        time.sleep(5)
else:
    raise Exception("Không thể kết nối tới Cassandra. Vui lòng chạy lại Cell 1 hoặc restart runtime.")

# Schema Nguồn (Gặp Hot Partition)
session.execute("CREATE KEYSPACE IF NOT EXISTS chat_system WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 1};")
session.set_keyspace("chat_system")
session.execute("""
    CREATE TABLE IF NOT EXISTS chat_table (
        room_id text, message_id timeuuid, user_id text, content text,
        msg_type text, device text, is_edited boolean, timestamp timestamp,
        PRIMARY KEY (room_id, message_id)
    ) WITH CLUSTERING ORDER BY (message_id DESC);
""")

# Schema Đích (Time-bucketing giải quyết Hot Partition)
session.execute("CREATE KEYSPACE IF NOT EXISTS chat_system_target WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 1};")
session.set_keyspace("chat_system_target")
session.execute("""
    CREATE TABLE IF NOT EXISTS chat_table_bucketed (
        room_id text, bucket_id text, message_id timeuuid, user_id text,
        content text, msg_type text, device text, is_edited boolean, timestamp timestamp,
        PRIMARY KEY ((room_id, bucket_id), message_id)
    ) WITH CLUSTERING ORDER BY (message_id DESC);
""")

print("Tạo Schema thành công!")

print("Đang sinh mock data...")
session.set_keyspace("chat_system")
sample_msgs = ["Dạ cảm ơn", "Shop còn hàng k?", "Mạng lag quá", "Alo 123"]
devices = ["ios", "android", "web"]
insert_query = session.prepare("INSERT INTO chat_table (room_id, message_id, user_id, content, msg_type, device, is_edited, timestamp) VALUES (?, now(), ?, ?, ?, ?, ?, ?)")

params = []
for _ in range(3000): # Sinh 3000 tin nhắn để demo nhanh
    params.append(("room_999", f"user_{random.randint(1,100)}", random.choice(sample_msgs), "text", random.choice(devices), False, datetime.now() - timedelta(days=random.randint(0,60))))

execute_concurrent_with_args(session, insert_query, params, concurrency=100)
print("Sinh mock data hoàn tất!")

## 3. PySpark ETL Migration (Phase 3)

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.datastax.spark:spark-cassandra-connector_2.12:3.5.0 pyspark-shell'

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, date_format

spark = SparkSession.builder \
    .appName("ETL_Colab") \
    .config("spark.cassandra.connection.host", "127.0.0.1") \
    .config("spark.cassandra.output.batch.size.bytes", "65536") \
    .config("spark.cassandra.output.concurrent.writes", "10") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

print("Đọc DataFrame từ Cassandra Nguồn...")
df_source = spark.read.format("org.apache.spark.sql.cassandra").options(table="chat_table", keyspace="chat_system").load()

# Transform: Thêm bucket_id yyyy-MM
df_transformed = df_source.withColumn("bucket_id", date_format(col("timestamp"), "yyyy-MM"))
df_transformed.show(5)

print("Thực thi luồng Ghi sang CSDL Đích...")
df_transformed.write.format("org.apache.spark.sql.cassandra") \
    .options(table="chat_table_bucketed", keyspace="chat_system_target") \
    .mode("append").save()
print("ETL Hoàn tất 100%!")

## 4. NLP Text Preprocessing (Phase 4)

In [ ]:
import re
from underthesea import word_tokenize

EMOTICONS_PATTERN = re.compile(r'(=[)(]+|:\)|:\(|<3|\?{2,}|!{2,})')
URL_PATTERN = re.compile(r'http[s]?://\S+|www\.\S+')
PUNCT_PATTERN = re.compile(r'[^\w\s]', flags=re.UNICODE)
SPACES_PATTERN = re.compile(r'\s+')

def clean_text(text):
    if not text: return ""
    text = text.lower()
    text = URL_PATTERN.sub('', text)
    
    emoticons_found = EMOTICONS_PATTERN.findall(text)
    for i, emo in enumerate(emoticons_found):
        text = text.replace(emo, f" EMO_{i} ", 1)

    text = PUNCT_PATTERN.sub('', text)
    text = SPACES_PATTERN.sub(' ', text).strip()

    tokens = word_tokenize(text, format="list")
    
    # Lọc stopwords đơn giản
    stopwords = {"thì", "là", "mà", "bị", "được", "quá", "cho", "hỏi"}
    cleaned_tokens = [t for t in tokens if t.startswith("EMO_") or (t not in stopwords and len(t) > 1)]
    
    final_text = " ".join(cleaned_tokens)
    for i, emo in enumerate(emoticons_found):
        final_text = final_text.replace(f"EMO_{i}", emo)
        
    return final_text

test_msgs = [
    "Mạng VNPT dạo này lag quá... https://vnpt.com.vn",
    "Cho e hỏi chi phí lắp wifi bao nhiêu ạ??? =)))"
]
for msg in test_msgs:
    print("RAW:", msg)
    print("CLN:", clean_text(msg), "\n")